# LLM Harness — step by step

Shows exactly how we call the LLM, using the same functions as the prototype
(`src/llm_compiler.py` + `src/mcp_server.py`):

1. Build the payload (per-slice telemetry + ML signals).
2. **No-tool** call: plain JSON (`call_llm_model`).
3. **Tool harness**: multi-round tool-calling (`call_llm_with_tools`), reconstructed
   step by step so you see the raw `tool_calls` and tool results.


In [1]:
%matplotlib inline
import sys, os, json
for _p in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.exists(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        os.chdir(_p)
        break

import numpy as np
from src import benchmark as bench
from src import capacity_model as cm
from src.dataset import load_result_dir, sorted_by_load
from src.kpi_predictor import MLPRegressor, violation_prob
from src.forecaster import LinearForecaster
from src.llm_compiler import build_payload, call_llm_model, SYSTEM_TOOL, _post
from src.mcp_server import call_tool, openai_tools

DATA = "datasets/generation_netwok_slicing_dataset/2.2.1-10kprocessed/results_slicing_WFQ_1000_routing_7800_7899"
MODEL = "glm-5.3-flash:cloud"


In [2]:
states, delays, losses = load_result_dir(DATA)
states, delays, losses = sorted_by_load(states, delays, losses)
cfgs = bench.build_cfgs(states, delays)
agg_nf = bench.build_agg_nf(states, delays)
agg = [sum(s.values()) for s in states]
predictor = MLPRegressor().fit(agg, delays)
fc = LinearForecaster(window=5)

i = 40
state = states[i]
obs = {"t": i, "slices": {s: {"load": state[s], "replicas": 1,
    "delay": cm.delay(state[s], 1, cfgs[s]), "loss": cm.loss(state[s], 1, cfgs[s])} for s in cm.SLICES}}
history = list(agg[:i+1])
forecast = fc.predict(history)
pv = violation_prob(float(predictor.predict([forecast])[0]), agg_nf.delay_budget)
payload = build_payload(obs, forecast, pv, cfgs)

print("per-slice load:", {s: round(state[s]) for s in cm.SLICES})
print("forecast:", round(forecast), "Mbps  p_violation:", round(pv, 3))
print("payload:")
print(json.dumps(payload, indent=2)[:600], "...")


per-slice load: {'eMBB': 126, 'URLLC': 164, 'mMTC': 5}
forecast: 297 Mbps  p_violation: 0.793
payload:
{
  "slices": {
    "eMBB": {
      "kpi_target": {
        "delay": 0.00639,
        "loss": 0.02
      },
      "kpi_now": {
        "delay": 0.01065,
        "loss": 0.2244897959183674
      },
      "load_mbps": 126.0,
      "replicas": 1,
      "max_replicas": 5
    },
    "URLLC": {
      "kpi_target": {
        "delay": 0.000852,
        "loss": 0.005
      },
      "kpi_now": {
        "delay": 0.00213,
        "loss": 0.38571428571428573
      },
      "load_mbps": 164.5,
      "replicas": 1,
      "max_replicas": 5
    },
    "mMTC": {
      "kpi_target": {
        "delay": 0.003195, ...


## 1. The system prompt

The same `SYSTEM_TOOL` string the harness sends as the system message.

In [3]:
print(SYSTEM_TOOL)

You are the Tier-2 compiler of a 6G network-slicing control plane. You receive per-slice telemetry (eMBB, URLLC, mMTC): load (Mbps), current replicas, current delay/loss, KPI targets, and ML signals. Use the provided tools to compute the correct replica count, then output ONLY a JSON object: {"actions": {"eMBB": <int>, "URLLC": <int>, "mMTC": <int>}, "reason": "<short>"}. Replica counts are integers in [1,5]. No prose outside the JSON.


## 2. The tools

`openai_tools` converts our MCP tool schemas into OpenAI function-calling
format. Here we expose `inspect_infrastructure` and `scale_replicas`.

In [4]:
tools = openai_tools(["inspect_infrastructure", "scale_replicas"])
print(json.dumps(tools, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "scale_replicas",
      "description": "Propose a validated replica count for a slice (enforces bounds).",
      "parameters": {
        "type": "object",
        "properties": {
          "slice_type": {
            "type": "string"
          },
          "replicas": {
            "type": "integer"
          }
        },
        "required": [
          "slice_type",
          "replicas"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "inspect_infrastructure",
      "description": "Computed infra state per slice: utilization, capacity, headroom, required replicas, KPI budgets.",
      "parameters": {
        "type": "object",
        "properties": {
          "state": {
            "type": "object"
          }
        },
        "required": [
          "state"
        ]
      }
    }
  }
]


## 3. No-tool call

`call_llm_model` — one HTTP round, the model answers JSON directly (no tools).

In [5]:
r = call_llm_model(MODEL, payload, cfgs)
print("raw response:", r["raw"])
print("actions:", r["actions"])
print("latency_ms:", round(r["latency_ms"], 1))

raw response: {"actions": {"eMBB": 3, "URLLC": 5, "mMTC": 1}, "reason": "URLLC severely violates delay/loss targets (2.5x/77x) -> scale to max; eMBB breaches delay/loss under 126Mbps -> scale to 3; mMTC well within KPIs -> hold at 1; high forecast+violation risk supports aggressive scaling."}
actions: {'eMBB': 3, 'URLLC': 5, 'mMTC': 1}
latency_ms: 2492.4


## 4. Tool harness, step by step

`call_llm_with_tools` does a multi-round loop. We reconstruct it here so you
see each round: the assistant asks for tool calls, we execute them against the
MCP server, feed the results back, and the model gives the final JSON.

In [6]:
messages = [
    {"role": "system", "content": SYSTEM_TOOL},
    {"role": "user", "content": json.dumps(payload)},
]
print("tool names sent:", [t["function"]["name"] for t in tools])

content = None
for round_ in range(4):
    data = _post(messages, tools=tools, model=MODEL)
    msg = data["choices"][0]["message"]
    print(f"\n=== round {round_+1} — finish_reason: "
          f"{data['choices'][0].get('finish_reason')} ===")
    if msg.get("tool_calls"):
        print("assistant tool_calls:")
        print(json.dumps(msg["tool_calls"], indent=2))
        messages.append({"role": "assistant", "content": msg.get("content") or "",
                         "tool_calls": msg["tool_calls"]})
        for tc in msg["tool_calls"]:
            fn = tc["function"]
            name = fn["name"]
            args = json.loads(fn.get("arguments") or "{}")
            result = call_tool(name, args, cfgs)
            print(f"--- execute {name}({args}) ->")
            print(json.dumps(result, indent=2))
            messages.append({"role": "tool", "tool_call_id": tc.get("id", f"c{round_}"),
                             "content": json.dumps(result)})
    else:
        content = msg.get("content") or ""
        print("final content:", content)
        break

if content:
    try:
        start, end = content.find("{"), content.rfind("}")
        actions = json.loads(content[start:end + 1])["actions"]
        print("actions:", actions)
    except Exception as e:
        print("could not parse JSON:", e)


tool names sent: ['scale_replicas', 'inspect_infrastructure']



=== round 1 — finish_reason: tool_calls ===
assistant tool_calls:
[
  {
    "id": "call_knazibe7",
    "index": 0,
    "type": "function",
    "function": {
      "name": "inspect_infrastructure",
      "arguments": "{\"state\":{\"ml\":{\"aggregate_forecast_mbps\":296.74999999999994,\"p_violation\":0.7928376123496519},\"slices\":{\"URLLC\":{\"kpi_now\":{\"delay\":0.00213,\"loss\":0.38571428571428573},\"kpi_target\":{\"delay\":0.000852,\"loss\":0.005},\"load_mbps\":164.5,\"replicas\":1},\"eMBB\":{\"kpi_now\":{\"delay\":0.01065,\"loss\":0.2244897959183674},\"kpi_target\":{\"delay\":0.00639,\"loss\":0.02},\"load_mbps\":126,\"replicas\":1},\"mMTC\":{\"kpi_now\":{\"delay\":0.0002582916160388822,\"loss\":0},\"kpi_target\":{\"delay\":0.003195,\"loss\":0.005},\"load_mbps\":5.000000000000002,\"replicas\":1}}}}"
    }
  }
]
--- execute inspect_infrastructure({'state': {'ml': {'aggregate_forecast_mbps': 296.74999999999994, 'p_violation': 0.7928376123496519}, 'slices': {'URLLC': {'kpi_now': {'del


=== round 2 — finish_reason: stop ===
final content: {"actions": {"eMBB": 2, "URLLC": 3, "mMTC": 1}, "reason": "eMBB delay/loss over budget and util 1.29 -> 2 replicas; URLLC delay 2.5x budget, util 1.63 -> 3; mMTC healthy, hold 1; p_violation 0.79 confirms scale-up"}
actions: {'eMBB': 2, 'URLLC': 3, 'mMTC': 1}
